# Squeezing gastruloids

This program was developed for “article” (authors, year) using [CellBasedModels.jl](https://github.com/dsb-lab/CellBasedModels.jl).

The report and the rest of the code can be found on [GitHub](https://github.com/).

## Preamble

### Packages

In [ ]:
println("Running on $(Threads.nthreads()) threads.")

using NBInclude
using DifferentialEquations
using CellBasedModels
using Distributions
using Random
try using GLMakie; Makie.inline!(true) catch; using CairoMakie end
using MathTeXEngine
using Printf
using Dates
using Glob

import CellBasedModels: update!

Makie.update_theme!(fonts = (regular = texfont(), bold = texfont(:bold), italic = texfont(:italic)))

In [ ]:
@nbinclude("preamble/functions-models.ipynb");

### Functions

In [ ]:
function initialize_growth(parameters; dt)

	com = Community(
		model,
		N = 1,
		dt = dt,
	)

	# Prameters
	for (par, val) in pairs(parameters)
		com[par] = val
	end

	# Initialization
	com.cell_state = 1
	com.x = 0.0
	com.y = 0.0
	com.z = 0.0
	com.vx = 0.0
	com.vy = 0.0
	com.vz = 0.0
	com.t_div = 1

	com.g_on = true
	com.d_on = false

	return com

end;


In [ ]:
function grow_size!(com, save_each, n_cells;
	n_control=100)

	loadToPlatform!(com, preallocateAgents = round(Int, n_cells * 1.1))
	i = 0
	n_total = 0

	while (com.N .< n_cells)
		if com.N > n_total
			println("N > $(n_total)")
			n_total += n_control
		end
		i += 1
		agentStepDE!(com)
		agentStepRule!(com)
		update!(com)
		computeNeighbors!(com)
		if i % save_each == 0
			saveRAM!(com)
		end
	end
	for i in 1:save_each
		i += 1
		agentStepDE!(com)
		update!(com)
	end
	saveRAM!(com)
	bringFromPlatform!(com)
end;


In [ ]:
function initialize_confined_diff!(com; 
    g_on = false,
    t_reset = true,
    r_agg = (maximum(com.x) - minimum(com.x)) * 0.5,
    center = 0.3
    )

    dt = com.dt
    
    r_center = center * r_agg
    for i in 1:com.N
        di0 = sqrt(com.x[i]^2 + com.y[i]^2 + com.z[i]^2)
        if di0 < r_center
            com.cell_state[i] = 3
        else com.cell_state[i] = 2
        end
    end

	com.g_on = g_on
    com.d_on = true

    if t_reset
        setfield!(com,:t, 0);
    end

    if g_on
        for i in 1:com.N
            t = com.t
            sample_right = 2 * com.tau_div[com.cell_state[i]] * com.sigma_div[] + dt
            for i in 1:com.N
                com.t_div[i] = t + CBMDistributions.uniform(dt, sample_right)
            end
        end
    end
    
	saveRAM!(com)

end;


In [ ]:
function differentiate_growing!(com, save_each, tf;
    preallocate = 2 * com.N * 2 ^ (tf / mean(com.tau_div))
    )

    if com.g_on[] == false
        println("You have to set g_on=true in initialize_diff!")
        return
    end
    message1 = "Possible numerical instabilities. 
            You might have to send argument preallocate = highnumber
            Default is preallocate = 2 * com.N * 2 ^ (tf / mean(com.tau_div))"
    message2 = "Numerical instabilities. 
            You might have to decrease the timestep"

	steps = round(Int64, tf / com.dt)
    step_control = steps / 10
    progress = step_control
    i = 0

    println("Initial N: $(com.N)")
    
    loadToPlatform!(com, preallocateAgents = round(Int, preallocate))
    
    for i in 1:steps
		if i > progress
			println("$(round(100 * i / steps))%")
			progress += step_control
		end
        agentStepDE!(com)
        agentStepRule!(com)
        update!(com)
        computeNeighbors!(com)
        if i % save_each == 0
            saveRAM!(com)
        end
        if all(com.N .> preallocate)
            println(message1)
            break
        end
    end        

	if i % save_each != 0
		saveRAM!(com)
    end
    
	bringFromPlatform!(com)

    println("Final N: $(com.N)")
    
end;


In [ ]:
function plot_pancake(com, color_map, mstart, mstop;
    boxsize = ((maximum(com.x) - minimum(com.x)) + com.r[1]) / 1.5,
    n = 4,
    showtime = false,
    shownumbers = true,
    filename = "pancake_plot.svg"
)

    fig = Figure(size = (n * 640, 600), figure_padding = 40)
    labelsize = 40
    
    d = getParameter(com, [:x, :y, :z, :r, :cell_state])

    for (i, pos) in enumerate(range(start = mstart, length = n, stop = mstop))
        pos = floor(Int, pos)
        println("Plot $i: timestamp $pos")

        t = round(com[pos].t, digits = 2)

        ax = Axis3(
            fig[1, i],
            aspect = :data,
            azimuth = 0,
            elevation = π/2,
            xlabel = "",
            ylabel = "",
            zlabel = "",
            xticklabelsize = labelsize,
            yticklabelsize = labelsize,
            zticklabelsize = labelsize,
            titlevisible = showtime,
            titlealign = :center,
            titlegap = 12,
            titlesize = labelsize,
            title = L"t=%$(t)"
        )

        if !shownumbers
            ax.xticklabelsize = 0
            ax.yticklabelsize = 0
            ax.zticklabelsize = 0
        end

        color = [color_map[j] for j in d[:cell_state][pos]]

        meshscatter!(
            ax,
            d[:x][pos],
            d[:y][pos],
            d[:z][pos],
            markersize = d[:r][pos],
            color = color
        )

        xlims!(ax, -boxsize, boxsize)
        ylims!(ax, -boxsize, boxsize)
        zlims!(ax, -boxsize, boxsize)
        
        ax.zlabel = ""
        ax.zticklabelsvisible = false
        ax.zgridvisible = false
    end

    display(fig)
	# save(filename, fig)
end

In [ ]:
function get_props(com)

	d = getParameter(com, [:t, :cell_state, :N])

	props = Dict()
	for state in 1:3  # 1=A, 2=B, 3=C
		props[state] = [sum(i .== state) for i in d[:cell_state]]
		props[state] = props[state] ./ d[:N]
	end

	return props

end;


In [ ]:
function plot_props(com, color_map, mstart, mstop, props)

    t1 = com[mstart].t
    t2 = com[mstop].t
    
	fig = Figure(resolution = (1000, 800), figure_padding = 25)
    labelsize = 50
		ax = Axis(
        fig[1, 1],
        xlabel = "Signalling time (h)",
        ylabel = "Proportion of cells", 
        xlabelsize = labelsize,
        ylabelsize = labelsize,
        xticklabelsize = labelsize,
        yticklabelsize = labelsize,
        aspect = 1,
        xticks = round.(range(t1, t2, 4), digits=1)
    )
	ylims!(ax, 0, 1)
	xlims!(ax, t1, t2)

	d = getParameter(com, [:t, :cell_state, :N])
	plots = []
	for i in 1:3
		p = lines!(ax, d[:t][mstart:mstop], props[i][mstart:mstop], color = color_map[i], linewidth = 5) # linestyle=:dash
		push!(plots, p)
	end
	labels = [L"state $A$", L"state $B$", L"state $C$"]
    Legend(
        fig[1, 2], 
        plots, labels, 
        labelsize = labelsize,
    )    
	display(fig)

end;


### Model

In [ ]:
model_new = ABM(3,

	# Global parameters
	model = Dict(
		# Mechanics
		:range => Float64,
		:mu => Array{Float64},
		:lambda => Float64,
		:f_rep => Array{Float64},
		:f_att => Array{Float64},
		:alpha_ecm => Float64,
		# Division
		:tau_div => Float64,
		:sigma_div => Float64,
		:olap => Float64,
		:g_on => Bool,
		# Differentiation
		:d_on => Bool,
		:b => Float64,
		:p => Float64,
		:q => Float64,
		:k => Float64,
		# Reference values
		:t0 => Float64,
		:r0 => Float64,
		:f0 => Float64,
		# Confinement
		:confinement => Bool,
		:L => Float64,
		:rep_wall => Float64,
	),


	# Agent parameters
	agent = Dict(
		:t_div => Float64,
		:ni => Int64,
		:cell_state => Int64,
		:r => Float64,
		# Mechanics
		:vx => Float64,
		:vy => Float64,
		:vz => Float64,
		:fx => Float64,
		:fy => Float64,
		:fz => Float64,
		# Protrusions
		:fpx => Float64,
		:fpy => Float64,
		:fpz => Float64,
		:marked => Bool,
		:t_paired => Float64,
		# Differentiation
		:ni_a => Float64,
		:r_ab => Float64,
		:r_bc => Float64,
	),


	# Mechanics
	agentODE = quote

		fx = 0
		fy = 0
		fz = 0
		vsum_x = 0
		vsum_y = 0
		vsum_z = 0
		ni = 0
		@loopOverNeighbors it2 begin
			dij = CBMMetrics.euclidean(x, x[it2], y, y[it2], z, z[it2])
			# Passive forces
			rij = 2 * r
			mu_ij = mu[cell_state, cell_state[it2]]
			if dij <= mu_ij*rij && dij > 0
				if dij < rij
					f_int = f_rep[cell_state, cell_state[it2]]
				else
					f_int = f_att[cell_state, cell_state[it2]]
				end
				fx += f_int * (rij / dij - 1) * (mu_ij * rij / dij - 1) * (x - x[it2]) / dij
				fy += f_int * (rij / dij - 1) * (mu_ij * rij / dij - 1) * (y - y[it2]) / dij
				fz += f_int * (rij / dij - 1) * (mu_ij * rij / dij - 1) * (z - z[it2]) / dij
			end
			# Counting neighbours
			if dij < range * rij
				ni += 1
				vsum_x += vx[it2]
				vsum_y += vy[it2]
				vsum_z += vz[it2]
			end
		end

		if confinement
			if z < r
				fz += rep_wall
			end

			diz_top = L - z
			if diz_top < r
				fz -= rep_wall
			end
		end

		# Equations of motion
		ni_relv_min = 5
		if ni < ni_relv_min
			inv = 1 / lambda
			vx = inv * fx
			vy = inv * fy
			vz = inv * fz
		else
			inv = 1 / (lambda * ni)
			vx = inv * (fx + vsum_x*alpha_ecm)
			vy = inv * (fy + vsum_y*alpha_ecm)
			vz = inv * (fz + vsum_z*alpha_ecm)
		end
		dt(x) = vx
		dt(y) = vy
		dt(z) = vz

	end,


	# Growth and differentiation
	agentRule = quote
		# Growth
		if g_on
			if t > t_div
				x_div = CBMDistributions.normal(0, 1)
				y_div = CBMDistributions.normal(0, 1)
				z_div = CBMDistributions.normal(0, 1)
				norm_div = sqrt(x_div^2 + y_div^2 + z_div^2)
				x_div /= norm_div
				y_div /= norm_div
				z_div /= norm_div

				r_sep = r * olap
				@addAgent(
					x = x + r_sep * x_div,
					y = y + r_sep * y_div,
					z = z + r_sep * z_div,
					vx = vx / 2,
					vy = vy / 2,
					vz = vz / 2,
					t_div = t + CBMDistributions.uniform(tau_div[cell_state] * (1 - sigma_div), tau_div * (1 + sigma_div))
				)
				@addAgent(
					x = x - r_sep * x_div,
					y = y - r_sep * y_div,
					z = z - r_sep * z_div,
					vx = vx / 2,
					vy = vy / 2,
					vz = vz / 2,
					t_div = t + CBMDistributions.uniform(tau_div[cell_state] * (1 - sigma_div), tau_div * (1 + sigma_div))
				)
				@removeAgent()
			end
		end

		# Differentiation
		if d_on == true && cell_state != 3
			ni = 0
			ni_a = 0
			@loopOverNeighbors it2 begin
				dij = CBMMetrics.euclidean(x, x[it2], y, y[it2], z, z[it2])
				if dij < range * 2 * r
					ni += 1
					if (cell_state[it2] == 1)
						ni_a += 1
					end
					# if (cell_state[it2] == 1)
					# 	ni_b += 1
					# end
					# if (cell_state[it2] == 1)
					# 	ni_c += 1
					# end
				end
			end

			if ni != 0
				ni_a /= ni
			end

			ran = CBMDistributions.uniform(0, 1)

			if cell_state == 1
				r_ab = p / (1 + k * ni_a)
				if ran < r_ab * dt
					cell_state = 2
				end

			elseif cell_state == 2
				r_bc = q / (1 + k * ni_a)
				if ran < r_bc * dt
					cell_state = 3
				end
			end
		end

	end, 
	
	
	agentAlg = CBMIntegrators.Heun(),
);

## Initalization

In [ ]:
# ORIGINAL MINE

parameters = Dict(
	:range => 1.2,
	:mu => 2 * [1 1 1; 1 1 1 ; 1 1 1],
	:r => 1,
	:lambda => 1,
	:tau_div => 5 * [1, 1, 1],
	:sigma_div => 0.5,
	:olap => 0.75,
	:p => 0.25,
	:q => 0.125,
	:k => 4.76,
	:b => 0.2,
	:f_rep => 2.5 * [1 1 1; 1 1 1 ; 1 1 1],
	:f_att => 		[1 1 1; 1 1 1 ; 1 1 1],
	:t0 => 2,
	:r0 => 5,
	:f0 => 20
);

model = model_new;
dt = 0.002;

In [ ]:
# # SOTIRIS

# parameters = Dict(
# 	:range => 1,		# changes	
# 	:mu => 
#         [1.55 1.5  1.525
#          1.5  1.7  1.45
#          2.0  2.0  2.0 ],
# 	:r => 1,
# 	:lambda => 1,
# 	:tau_div => [300, 300, 750],	# changes
# 	:sigma_div => 0.5,	
# 	:olap => 0.75,		# changes
# 	:p => 0.009333,		# changes
# 	:q => 0.004667,		# changes
# 	:k => 5,			# changes
# 	:b => 0.2,	
# 	# changes
# 	:f_rep => 
#         [3 3 3. ; 
#          3 3 3. ; 
#          3 3 5.1],
# 	:f_att => 
#         [6   6   2.4; 
#          6   6   1.6; 
#          2.4 1.6 4.8],
# 	:t0 => 2,
# 	:r0 => 5,
# 	:f0 => 20
# );

# model = model_new;
# dt = 0.02;

In [ ]:
save_each = round(Int64, 0.25 / dt);            # save community once every save_each instances
n_cells = 300;                                  # desired number of cells

Random.seed!(2345)                              # plant seed for reproducibility
com = initialize_growth(parameters; dt = dt);   # initialization. dt can also be set directy

com.confinement = true
layers = 1
com.L = (1 * (2*com.r)) * layers
com.rep_wall = 50;

com.alpha_ecm = 0.9 * com.lambda


grow_size!(com, save_each, n_cells)       # grow for a given n_cells
m0 = length(com);

growncom = deepcopy(com);               # backup to go back

println(com.N)
# println(formed_correctly(com))          # no sparse cells caused by numerical errors
plot_pancake(com, color_map, 1, m0; filename = "pancake_diff.png")

## Differentiation

In [ ]:
# com = deepcopy(growncom)
# Random.seed!(2345);

# com.tau_div = [300, 300, 750]
# com.p = 0.009333
# com.q = 0.004667

# setfield!(com,:dt, 0.02)

In [ ]:
initialize_confined_diff!(com, g_on=true)                   # (...; g_on=false, treset=true)
m1 = length(com);

differentiate_growing!(com, save_each, 10)      # (...; prot=true, fp=200, kp_on=0.7, kp_off=0.4)
m2 = length(com);

# evolvedcom = deepcopy(com);

plot_pancake(com, color_map, m1, m2; filename = "pancake_growth.svg")

## Proportions

In [ ]:
# props = get_props(com);      # compute proportion of each state for the saved timestamps
# plot_props(com, color_map, m1, m2, props)